In [1]:
from pathlib import Path
import os
import sys

# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.query import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *

EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully
EU1_PROD_Conn created successfully
EU2_PROD_Conn created successfully


In [11]:
query = Query("""SELECT Customer.Id as CustomerId, Name as Name,
                Location.Latitude as Latitude,
                Location.Longitude as Longitude,
                Location.Description as Location
                FROM Customer 
                LEFT JOIN Location ON Customer.Id = Location.CustomerId
                WHERE Active = 1""")
customer_eu1 = query.execute(EU1_Conn)
customer_eu1['DBLocation'] = 'EU1'
customer_eu2 = query.execute(EU2_Conn)
customer_eu2['DBLocation'] = 'EU2'
customer_df = pd.concat([customer_eu1, customer_eu2])

customer_df = customer_df[~customer_df['Name'].str.upper().str.contains('PICARRO', case=False) & ~customer_df['Name'].str.upper().str.contains('TEST', case=False)]
customer_df = customer_df[~customer_df['Name'].str.strip().str.upper().eq('ITALGAS-SERVICE')]


In [12]:
import reverse_geocoder as rg

ISO3166 = {
    "AD": "Andorra", "AL": "Albania", "AT": "Austria", "BA": "Bosnia and Herzegovina",
    "BE": "Belgium", "BG": "Bulgaria", "BY": "Belarus", "CH": "Switzerland",
    "CY": "Cyprus", "CZ": "Czechia", "DE": "Germany", "DK": "Denmark",
    "EE": "Estonia", "ES": "Spain", "FI": "Finland", "FR": "France",
    "GB": "United Kingdom", "GR": "Greece", "HR": "Croatia", "HU": "Hungary",
    "IE": "Ireland", "IS": "Iceland", "IT": "Italy", "LI": "Liechtenstein",
    "LT": "Lithuania", "LU": "Luxembourg", "LV": "Latvia", "MC": "Monaco",
    "MD": "Moldova", "ME": "Montenegro", "MK": "North Macedonia", "MT": "Malta",
    "NL": "Netherlands", "NO": "Norway", "PL": "Poland", "PT": "Portugal",
    "RO": "Romania", "RS": "Serbia", "RU": "Russia", "SE": "Sweden",
    "SI": "Slovenia", "SK": "Slovakia", "SM": "San Marino", "UA": "Ukraine",
    "VA": "Vatican City", "XK": "Kosovo",
}

located = customer_df.dropna(subset=["Latitude", "Longitude"]).copy()
if not located.empty:
    results = rg.search(
        list(zip(located["Latitude"].astype(float), located["Longitude"].astype(float))),
        mode=1,
    )
    located["Country"] = [ISO3166.get(r["cc"], r["cc"]) for r in results]
    country_by_customer = located.groupby("CustomerId")["Country"].agg(
        lambda s: s.mode().iloc[0] if not s.mode().empty else None
    )
else:
    country_by_customer = pd.Series(dtype=object)

customer_df = (
    customer_df.groupby(["CustomerId", "Name", "DBLocation"], as_index=False)
    .agg(
        Latitude=("Latitude", "mean"),
        Longitude=("Longitude", "mean"),
        LocationCount=("CustomerId", "size"),
    )
)
customer_df["Country"] = customer_df["CustomerId"].map(country_by_customer)
customer_df

,CustomerId,Name,Latitude,Longitude,Location,DBLocation
0,A1E8BEC0-6A89-9454-137B-3A0A41825A09,NBB,52.528749,13.409796,"Berlin, Germany",EU1
1,8AABFD1C-862E-CF1E-5545-3A0EF16CBB61,ENBW,48.791288,9.217140,NetzeBW,EU1
2,8AABFD1C-862E-CF1E-5545-3A0EF16CBB61,ENBW,48.783039,9.181932,SN - Stuttgart Netze,EU1
3,8AABFD1C-862E-CF1E-5545-3A0EF16CBB61,ENBW,49.142952,9.207282,HNVG - Stadtwerke Heilbronn,EU1
4,8AABFD1C-862E-CF1E-5545-3A0EF16CBB61,ENBW,48.939377,8.378666,NGS - Netze Südwest,EU1
...,...,...,...,...,...,...
34,55C8203E-E789-03B8-FB19-3A2041A8F1C2,Retragas,45.520062,10.215555,Brescia,EU2
35,5F35434A-3029-76C8-4DDA-3A2205A7096F,GERGAS,42.790085,11.094079,Grosseto,EU2
36,0A9052C7-ABBF-9009-2575-3A2205BF1798,EDMA,43.614917,13.522765,Ancona,EU2
37,5B684DC1-AB8A-FAF9-C333-3A2205D870E4,AES Fano,43.838145,13.007247,Fano,EU2


In [ ]:
for _, row in customer_df.iterrows():
    customer_name = row["Name"]
    db_location = row["DBLocation"]
    country = row["Country"] if pd.notna(row["Country"]) else None
    conn = EU1_Conn if db_location == "EU1" else EU2_Conn
    add_customer(customer_name, conn, country=country)

In [3]:
print(DB_PATH)

database/KPIHub_EuropeAvg.db


In [4]:
KPI_Customer.query_table(arguments = {'db_path': DB_PATH})

,CustomerId,Name,ShortName,Active,DBLocation,LastUpdated
0,A1E8BEC0-6A89-9454-137B-3A0A41825A09,NBB,NBB,1,EU1,2026-08-13 19:26:31.594489
1,8AABFD1C-862E-CF1E-5545-3A0EF16CBB61,ENBW,ENBW,1,EU1,2026-08-13 19:26:31.614683
2,E5F6C480-6796-9A6E-79DE-3A0EF1744B1C,E-NETZ SUDHESSEN,E-NETZSUDHESSEN,1,EU1,2026-08-13 19:26:31.632771
3,57696369-C382-1033-0237-3A10E44BB5CF,Schwabennetz,Schwabennetz,1,EU1,2026-08-13 19:26:31.649136
4,7D9E66E1-6EE8-27D6-76B4-3A10E451DD30,EWE,EWE,1,EU1,2026-08-13 19:26:31.661585
...,...,...,...,...,...,...
75,55C8203E-E789-03B8-FB19-3A2041A8F1C2,Retragas,Retragas,1,EU2,2026-08-13 19:26:32.422862
76,5F35434A-3029-76C8-4DDA-3A2205A7096F,GERGAS,GERGAS,1,EU2,2026-08-13 19:26:32.431534
77,0A9052C7-ABBF-9009-2575-3A2205BF1798,EDMA,EDMA,1,EU2,2026-08-13 19:26:32.438968
78,5B684DC1-AB8A-FAF9-C333-3A2205D870E4,AES Fano,AESFano,1,EU2,2026-08-13 19:26:32.447242
